In [0]:
import websocket
import json
from pyspark.sql import Row

In [0]:
ws = websocket.create_connection("wss://ws.kraken.com/v2")

subscribe_message = {
    "method": "subscribe",
    "params": {
        "channel": "trade",
        "symbol": ["BTC/USD"]
    }
}

ws.send(json.dumps(subscribe_message))

print("Connected to Kraken WebSocket")

In [0]:
message = ws.recv()

print(message)

In [0]:
while True:
    message = ws.recv()
    data = json.loads(message)

    if data.get("channel") == "trade":
        print(data)

In [0]:
message = ws.recv()
data = json.loads(message)

if data.get("channel") == "trade":
    trades = data.get("data", [])

    for trade in trades:
        print(
            trade["symbol"],
            trade["side"],
            trade["price"],
            trade["qty"],
            trade["trade_id"],
            trade["timestamp"]
        )

In [0]:
trade_rows = []

while len(trade_rows) < 20:
    message = ws.recv()
    data = json.loads(message)

    if data.get("channel") == "trade":
        for trade in data.get("data", []):
            trade_rows.append((
                trade["symbol"],
                trade["side"],
                float(trade["price"]),
                float(trade["qty"]),
                int(trade["trade_id"]),
                trade["ord_type"],
                trade["timestamp"]
            ))

            if len(trade_rows) >= 20:
                break

print("Trades collected:", len(trade_rows))

In [0]:
columns = [
    "symbol",
    "side",
    "price",
    "qty",
    "trade_id",
    "ord_type",
    "event_timestamp"
]

trades_df = spark.createDataFrame(trade_rows, columns)

trades_df.show(20, truncate=False)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    TimestampType
)

bronze_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("side", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("qty", DoubleType(), False),
    StructField("trade_id", LongType(), False),
    StructField("ord_type", StringType(), False),
    StructField("event_timestamp", TimestampType(), False)
])

In [0]:
from datetime import datetime

trade_rows_fixed = []

for row in trade_rows:
    trade_rows_fixed.append((
        row[0],
        row[1],
        row[2],
        row[3],
        row[4],
        row[5],
        datetime.fromisoformat(row[6].replace("Z", "+00:00"))
    ))

In [0]:
bronze_df = spark.createDataFrame(
    trade_rows_fixed,
    bronze_schema
)

bronze_df.printSchema()

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze_crypto_trades")

In [0]:
spark.sql("""
SELECT *
FROM bronze_crypto_trades
ORDER BY event_timestamp
""").show(20, truncate=False)

In [0]:
%sql
SELECT
    trade_id,
    COUNT(*) AS record_count
FROM bronze_crypto_trades
GROUP BY trade_id
HAVING COUNT(*) > 1;

In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_df_enriched = (
    bronze_df
    .withColumn("source_system", lit("Kraken"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

bronze_df_enriched.printSchema()

In [0]:
bronze_df_enriched.show(5, truncate=False)

In [0]:
spark.sql("DROP TABLE IF EXISTS bronze_crypto_trades")

In [0]:
bronze_df_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_crypto_trades")

In [0]:
spark.sql("""
SELECT *
FROM bronze_crypto_trades
ORDER BY event_timestamp
""").show(20, truncate=False)